B20. Эффект модератора отдельно от эффекта состава.....


In [1]:
import sys, os, re

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/panel_mechanisms')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
import statsmodels.api as sm
A = load_arms()
ITEMS, POS = load_items()
print()
ORGANIZER = {'P-base': 'gpt', 'P-weak': 'qwen', 'P-mixed': 'deepseekv4', 'P-strong': 'deepseekv4', 'homog gpt': 'gpt', 'homog qwen': 'qwen', 'homog deepseekv4': 'deepseekv4', 'homog gemini': 'gemini'}
PANEL_REL = {'P-base': PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv', 'P-weak': PANELS_OTHER_DIR['P-weak'] + '/llm_as_judge/' + PANEL_FILE['P-weak'], 'P-mixed': PANELS_OTHER_DIR['P-mixed'] + '/llm_as_judge/' + PANEL_FILE['P-mixed'], 'P-strong': PANELS_OTHER_DIR['P-strong'] + '/llm_as_judge/' + PANEL_FILE['P-strong'], 'homog gpt': HOMOG_DIRS['gpt'] + '/llm_as_judge/delphi_homog_gpt_noroles_judge.csv', 'homog qwen': HOMOG_DIRS['qwen'] + '/llm_as_judge/delphi_homog_qwen_noroles_judge.csv', 'homog gemini': HOMOG_DIRS['gemini'] + '/llm_as_judge/delphi_homog_gemini_noroles_judge.csv', 'homog deepseekv4': HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'}


плеч загружено: 18
   baseline gpt, baseline qwen, baseline deepseekv4, baseline deepseekv4pro, baseline gemini, baseline glm, baseline kimi, USC@3 gpt, USC@5 gpt, USC@7 gpt, homog gpt, homog qwen, homog gemini, homog deepseekv4, P-weak, P-mixed, P-strong, P-base


плеч с попунктными вердиктами: 18
общих положительных пунктов: 2249



In [2]:
def synth_lift(pname):
    comp = COMPOSITION[pname]
    members = ['baseline ' + m for m in comp]
    if pname not in ITEMS or not all((m in ITEMS for m in members)):
        return None
    rows = []
    for k in POS:
        pts = ITEMS[pname][k]['points']
        v = [int(ITEMS[m][k]['met']) for m in members]
        rows.append({'case': k[0], 'points': pts, 'panel': int(ITEMS[pname][k]['met']), 'maj': int(sum(v) >= 2), 'union': int(sum(v) >= 1), 'spread': int(0 < sum(v) < 3)})
    D = pd.DataFrame(rows)
    agg = D.groupby('case').apply(lambda g: pd.Series({'panel': (g['panel'] * g['points']).sum() / g['points'].sum(), 'maj': (g['maj'] * g['points']).sum() / g['points'].sum(), 'union': (g['union'] * g['points']).sum() / g['points'].sum(), 'разногласий': g['spread'].mean()}), include_groups=False)
    agg['прирост синтеза'] = agg['panel'] - agg['maj']
    agg['прирост над оракулом'] = agg['panel'] - agg['union']
    return agg
LIFT = {p: synth_lift(p) for p in PANELS if p in A}
LIFT = {k: v for k, v in LIFT.items() if v is not None}
singles = {m.replace('baseline ', ''): float(js(A, m).mean()) for m in A if m.startswith('baseline ')}
rows = []
for p, g in LIFT.items():
    d = rd(PANEL_REL[p])
    syn_w = d['winner_protocol'].map(words) if 'winner_protocol' in d.columns else pd.Series(dtype=float)
    nq = pd.to_numeric(d.get('organizer_after_r1_n_unified_questions'), errors='coerce')
    syn_tok = pd.to_numeric(d.get('final_synthesis_total_tokens'), errors='coerce')
    org_tok = pd.to_numeric(d.get('organizer_after_r1_total_tokens'), errors='coerce').fillna(0) + pd.to_numeric(d.get('organizer_after_r2_total_tokens'), errors='coerce').fillna(0) + syn_tok.fillna(0)
    rows.append({'панель': p, 'организатор': ORGANIZER[p], 'гетерогенная': bool(HETERO[p]), 'сила организатора': singles.get(ORGANIZER[p], np.nan), 'ср. сила участников': float(np.mean([singles.get(m, np.nan) for m in COMPOSITION[p]])), 'панель': float(g['panel'].mean()), 'большинство': float(g['maj'].mean()), 'прирост синтеза': float(g['прирост синтеза'].mean()), 'прирост над оракулом': float(g['прирост над оракулом'].mean()), 'доля спорных пунктов': float(g['разногласий'].mean()), 'слов в синтезе': float(syn_w.mean()) if len(syn_w) else np.nan, 'вопросов сведено': float(nq.mean()) if nq is not None else np.nan, 'токенов организатора': float(org_tok.mean())})
M = pd.DataFrame(rows).set_index('панель')
show(M[['организатор', 'гетерогенная', 'сила организатора', 'ср. сила участников', 'большинство', 'прирост синтеза', 'прирост над оракулом']], 'Вклад организатора по панелям')



Вклад организатора по панелям
------------------------------------------------------------
       организатор  гетерогенная  сила организатора  ср. сила участников  большинство  прирост синтеза  прирост над оракулом
панель                                                                                                                      
0.7900         gpt          True             0.6258               0.6094       0.6295           0.1605                0.0211
0.7532        qwen          True             0.5622               0.5157       0.5340           0.2192                0.0695
0.7920  deepseekv4          True             0.6401               0.5506       0.5734           0.2186                0.0670
0.8083  deepseekv4          True             0.6401               0.6176       0.6304           0.1779                0.0350
0.7635         gpt         False             0.6258               0.6258       0.6407           0.1228                0.1228
0.7037        qwen         False 

In [3]:
same_org = M[M['организатор'] == 'deepseekv4']
show(same_org[['ср. сила участников', 'большинство', 'прирост синтеза']], 'Один организатор (deepseekv4), разные составы')
print()
print('размах прироста синтеза ВНУТРИ организатора deepseekv4: %.4f' % (same_org['прирост синтеза'].max() - same_org['прирост синтеза'].min()))
print('размах прироста синтеза МЕЖДУ организаторами (все панели): %.4f' % (M['прирост синтеза'].max() - M['прирост синтеза'].min()))
print()
by_org = M.groupby('организатор')['прирост синтеза'].agg(['size', 'mean', 'std'])
show(by_org, 'Прирост синтеза по организаторам')
if 'P-mixed' in LIFT and 'P-strong' in LIFT:
    a = LIFT['P-mixed']['прирост синтеза']
    b = LIFT['P-strong']['прирост синтеза']
    ix = a.index.intersection(b.index)
    d = a.loc[ix] - b.loc[ix]
    print()
    print('P-mixed против P-strong (общий организатор), покейсно:')
    print('   Δ прироста синтеза = %+.4f, SE = %.4f, p = %s' % (d.mean(), se_clt(d), fmt_p(stats.wilcoxon(d)[1])))
    print('   Разница здесь может идти только от состава, потому что')
    print('   организатор один и тот же.')



Один организатор (deepseekv4), разные составы
------------------------------------------------------------
        ср. сила участников  большинство  прирост синтеза
панель                                                   
0.7920               0.5506       0.5734           0.2186
0.8083               0.6176       0.6304           0.1779
0.7832               0.6401       0.6510           0.1322

размах прироста синтеза ВНУТРИ организатора deepseekv4: 0.0864
размах прироста синтеза МЕЖДУ организаторами (все панели): 0.1437


Прирост синтеза по организаторам
------------------------------------------------------------
             size   mean    std
организатор                    
deepseekv4      3 0.1762 0.0432
gemini          1 0.2665    NaN
gpt             2 0.1416 0.0267
qwen            2 0.1727 0.0658

P-mixed против P-strong (общий организатор), покейсно:
   Δ прироста синтеза = +0.0407, SE = 0.0121, p = 9.7e-04
   Разница здесь может идти только от состава, потому что
   организат

In [4]:
X = M.dropna(subset=['сила организатора', 'ср. сила участников'])
rows = []
for spec in (['сила организатора'], ['ср. сила участников'], ['сила организатора', 'ср. сила участников'], ['ср. сила участников', 'доля спорных пунктов']):
    Xi = sm.add_constant(X[spec].astype(float))
    m = sm.OLS(X['прирост синтеза'].astype(float), Xi).fit()
    for k in spec:
        rows.append({'спецификация': ' + '.join(spec), 'предиктор': k, 'коэф': m.params[k], 'p': m.pvalues[k], 'R²': m.rsquared, 'n': int(m.nobs)})
show(pd.DataFrame(rows).set_index(['спецификация', 'предиктор']), 'Прирост синтеза ~ характеристики панели (n = 8, разведочно)')
print()
print('ПРЕДУПРЕЖДЕНИЕ. При n = 8 и двух предикторах остаётся 5 степеней')
print('свободы. Доверять p здесь нельзя; смотреть надо на знак и на то,')
print('меняется ли он между спецификациями.')



Прирост синтеза ~ характеристики панели (n = 8, разведочно)
------------------------------------------------------------
                                                                   коэф      p     R²  n
спецификация                               предиктор                                    
сила организатора                          сила организатора    -0.4415 0.0961 0.3932  8
ср. сила участников                        ср. сила участников  -0.6185 0.0078 0.7194  8
сила организатора + ср. сила участников    сила организатора     0.5550 0.1294 0.8308  8
                                           ср. сила участников  -1.1392 0.0156 0.8308  8
ср. сила участников + доля спорных пунктов ср. сила участников  -0.6399 0.0027 0.8741  8
                                           доля спорных пунктов  0.1257 0.0559 0.8741  8

ПРЕДУПРЕЖДЕНИЕ. При n = 8 и двух предикторах остаётся 5 степеней
свободы. Доверять p здесь нельзя; смотреть надо на знак и на то,
меняется ли он между спецификациями

In [5]:
rows = []
for p, g in LIFT.items():
    s = g.dropna(subset=['разногласий', 'прирост синтеза'])
    if len(s) < 30:
        continue
    r, pv = stats.spearmanr(s['разногласий'], s['прирост синтеза'])
    Xi = sm.add_constant(s[['разногласий']].astype(float))
    m = sm.OLS(s['прирост синтеза'].astype(float), Xi).fit(cov_type='HC3')
    rows.append({'панель': p, 'организатор': ORGANIZER[p], 'кейсов': len(s), 'rho (Спирмен)': r, 'p': pv, 'коэф регрессии': m.params['разногласий'], 'p регрессии': m.pvalues['разногласий']})
show(pd.DataFrame(rows).set_index('панель'), 'Связь «разногласия участников → прирост синтеза» внутри панелей')
print()
print('Положительная связь = организатор действительно работает разрешением')
print('разногласий. Отсутствие связи = он добавляет ценность независимо от')
print('того, спорили участники или нет, то есть его вклад — оформление и')
print('дописывание, а не арбитраж. Второе согласуется с §3.4.')



Связь «разногласия участников → прирост синтеза» внутри панелей


---------------------------------------------------------------
                 организатор  кейсов  rho (Спирмен)      p  коэф регрессии  p регрессии
панель                                                                                 
P-base                   gpt     220         0.0701 0.3008          0.0521       0.3587
P-weak                  qwen     220         0.0025 0.9708         -0.0165       0.8268
P-mixed           deepseekv4     220         0.0544 0.4224          0.0750       0.3409
P-strong          deepseekv4     220         0.1371 0.0421          0.1375       0.0537
homog gpt                gpt     220            NaN    NaN          0.0000          NaN
homog qwen              qwen     220            NaN    NaN          0.0000          NaN
homog deepseekv4  deepseekv4     220            NaN    NaN          0.0000          NaN
homog gemini          gemini     220            NaN    NaN          0.0000          NaN

Положительная связь = организатор действительно работа

In [6]:
show(M[['организатор', 'слов в синтезе', 'вопросов сведено', 'токенов организатора', 'прирост синтеза']].sort_values('прирост синтеза', ascending=False), 'Продукция организатора')
sub = M.dropna(subset=['слов в синтезе', 'прирост синтеза'])
if len(sub) >= 4:
    r, p = stats.spearmanr(sub['слов в синтезе'], sub['прирост синтеза'])
    print('\nсвязь «длина синтеза → прирост»: rho = %+.3f, p = %s (n = %d)' % (r, fmt_p(p), len(sub)))
    r2, p2 = stats.spearmanr(sub['токенов организатора'], sub['прирост синтеза'])
    print('связь «токены организатора → прирост»: rho = %+.3f, p = %s' % (r2, fmt_p(p2)))



Продукция организатора
------------------------------------------------------------
       организатор  слов в синтезе  вопросов сведено  токенов организатора  прирост синтеза
панель                                                                                     
0.7225      gemini        429.9389            7.9301             6289.6769           0.2665
0.7532        qwen        454.7205           10.1092             9475.2533           0.2192
0.7920  deepseekv4        549.1266            5.8122            11426.6070           0.2186
0.8083  deepseekv4        557.1092            5.1616            12068.6987           0.1779
0.7900         gpt        478.5371            9.1703            12209.4410           0.1605
0.7832  deepseekv4        551.0393            8.5066            12456.4934           0.1322
0.7037        qwen        447.9694           10.0218              226.6114           0.1262
0.7635         gpt        478.3057            8.0611            10130.3624           0.

In [7]:
hom = M[~M['гетерогенная']]
het = M[M['гетерогенная']]
cmp_ = pd.DataFrame({'гомогенные (n=%d)' % len(hom): hom[['большинство', 'прирост синтеза', 'прирост над оракулом', 'доля спорных пунктов']].mean(), 'гетерогенные (n=%d)' % len(het): het[['большинство', 'прирост синтеза', 'прирост над оракулом', 'доля спорных пунктов']].mean()})
cmp_['разница'] = cmp_.iloc[:, 1] - cmp_.iloc[:, 0]
show(cmp_, 'Гомогенные против гетерогенных')
print()
print('У гомогенных панелей прирост синтеза положителен и велик — при том,')
print('что разнообразия состава там нет вовсе. Это прямое доказательство,')
print('что ценность создаётся ПРОЦЕДУРОЙ, а не разнообразием мнений.')



Гомогенные против гетерогенных
------------------------------------------------------------
                      гомогенные (n=4)  гетерогенные (n=4)  разница
большинство                     0.5813              0.5918   0.0105
прирост синтеза                 0.1619              0.1940   0.0321
прирост над оракулом            0.1619              0.0481  -0.1138
доля спорных пунктов            0.0000              0.3081   0.3081

У гомогенных панелей прирост синтеза положителен и велик — при том,
что разнообразия состава там нет вовсе. Это прямое доказательство,
что ценность создаётся ПРОЦЕДУРОЙ, а не разнообразием мнений.


In [8]:
save(M.reset_index(), 'organizer_effect_and_panel_composition_moderator.csv')
save(pd.concat([g.assign(панель=p) for p, g in LIFT.items()]).reset_index(), 'organizer_effect_and_panel_composition_lift_by_case.csv')


сохранено: out/organizer_effect_and_panel_composition_moderator.csv  (8 строк)


сохранено: out/organizer_effect_and_panel_composition_lift_by_case.csv  (1760 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\panel_mechanisms\\results\\organizer_effect_and_panel_composition_lift_by_case.csv'